# 🎓 WE4 · Notebook 06 — Safe / Risk-Aware RL
## The scenic route that was profitable right up until it wasn't

**The setting.** GaGGle Alpine runs one autonomous delivery robot. Every day it carries supplies
from a village depot to a mountain lodge, across a small mountain map with several possible
routes. Some routes earn more (tourists tip on the scenic stretches), and some can end with the
robot at the bottom of a gorge. Last quarter it did — which is why the company now has an
**insurance contract**: the insurer keeps covering the robot only if the **expected damage per
trip stays under an agreed cap**. So there are two numbers to manage, not one: profit, which you
want high, and damage, which the contract caps.

**How we will work.** GaGGle keeps a **digital twin** of the mountain: a simulator that contains
the map, the payouts, and the exact probability that each kind of ground gives way. Everything in
this notebook runs against this model. That means no trial-and-error learning is needed — we can
**plan** directly, with a standard solver, and every number you see can be computed exactly.

| Part | What happens |
|---|---|
| 0 | Two offers with identical expected value — you pick one, and that choice is the whole topic |
| 1 | The delivery problem written as a **constrained MDP**: reward, cost, and the budget `d` |
| 2 | The bluntest safety tool first: a hand-made **shield**, applied to an off-the-shelf planner |
| 3 | Choosing a route under the contract: the budget `d`, the internal price `λ`, the loop that finds `λ` from `d`, and the tail question `α` |

The same four ideas appear at production scale in current research: constrained policy
optimisation against a learned safety critic
([Leng et al. 2025](https://arxiv.org/abs/2503.19690)) and risk-sensitive distributional RL
([Xiao, Yu & Ying 2025](https://arxiv.org/abs/2405.14749)).

## 0. Setup

No manual uploads needed: the first cell pulls the exercise files (including `risk_viz.py`,
which contains the simulator and the display helpers) directly from GitHub, and the second
installs the small set of requirements.
*(While the course repo is private, Colab needs a `GITHUB_TOKEN` in its Secrets panel — your
instructors will have handled this before you ever see this notebook.)*

Run the three cells below before anything else.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE4-public"
HELPER     = os.path.join("6_risk_rl", "exercise", "risk_viz.py")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    token = ""
    try:                                  # private repo (testing): read token from Secrets
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:                     # public repo (students): no token needed
        token = ""
    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding `6_risk_rl/exercise/` — so imports resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()),
              os.path.dirname(os.path.dirname(os.getcwd())), os.getcwd()]:
    if os.path.exists(os.path.join(_root, HELPER)):
        os.chdir(_root)
        break
else:
    # Last resort (public repo, git unavailable): fetch just the two files we need.
    try:
        from urllib.request import urlretrieve
        os.makedirs(os.path.dirname(HELPER), exist_ok=True)
        base = f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/main/6_risk_rl/exercise/"
        for fname in ("risk_viz.py", "requirements_risk.txt"):
            urlretrieve(base + fname, os.path.join("6_risk_rl", "exercise", fname))
        print("Fetched the exercise files directly.")
    except Exception:
        raise FileNotFoundError(
            "Could not find the repo (6_risk_rl/exercise/risk_viz.py). If it is still private, add a "
            "GITHUB_TOKEN secret (see the note above) and re-run this cell.")
sys.path.insert(0, os.path.join(os.getcwd(), "6_risk_rl", "exercise"))   # make helpers importable
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r 6_risk_rl/exercise/requirements_risk.txt

**0.3 — Import the libraries.** The world, the widgets and the quizzes live in **`risk_viz`**
so the teaching cells stay about the *idea*, not about HTML.

> 📦 **You do not need to read `risk_viz.py`.** It holds the simulator and the drawing code;
> every number that matters is printed in the notebook itself.

In [ ]:
import numpy as np

import importlib
import risk_viz as rv
importlib.reload(rv)       # pick up the latest helpers even if this cell reruns

rng = np.random.default_rng(0)
print("Ready. The robot is charged. 🤖")

---
# Part 0 — Before anything else: one question

Forget mountains for a moment. **Two offers are on your desk**, and you must take exactly one.

- **Offer A — the steady one.** It pays **7, 9, 11 or 13 M CHF** a year, each equally likely.
- **Offer B — the bold one.** It pays **25 M** in four years out of five. In the fifth year it
  *loses* **50 M**.

Before you read on: which one do you want?

In [ ]:
rv.two_offers()

### Check the arithmetic yourself: the means really are identical

$$\underbrace{\frac{7 + 9 + 11 + 13}{4}}_{\text{Offer A}} \;=\; +10
\qquad\qquad
\underbrace{0.8 \times (+25) \;+\; 0.2 \times (-50)}_{\text{Offer B}} \;=\; 20 - 10 \;=\; +10$$

Over a long run the two offers pay exactly the same.

In [ ]:
offer_A = {7: 0.25, 9: 0.25, 11: 0.25, 13: 0.25}
offer_B = {25: 0.80, -50: 0.20}

mean = lambda d: sum(payout * prob for payout, prob in d.items())
print(f"Offer A — expected value : {mean(offer_A):+.1f} M CHF per year")
print(f"Offer B — expected value : {mean(offer_B):+.1f} M CHF per year")
print(f"identical? {mean(offer_A) == mean(offer_B)}")
print()
print(f"Offer A — worst year     : {min(offer_A):+.0f} M")
print(f"Offer B — worst year     : {min(offer_B):+.0f} M   <- and it arrives 1 year in 5")

In [ ]:
rv.mc_quiz("two_offers")

Expected value, the quantity every optimiser you have met so far maximises, scores these two
offers identically. Almost nobody reading them is indifferent, though: something in you priced
that −50 differently.

An algorithm prices it that way only if you tell it to, and that is what the rest of the notebook
does. Part 1 writes risk down as a separate quantity, Part 2 tries the bluntest fix, Part 3 builds
the real dials.

---
# Part 1 — The delivery problem, written as a constrained MDP

*No code to write in this part. The goal is to understand the world and the objective; the
programming starts in Part 2.*

In [ ]:
rv.the_briefing()

## 1.1 · The six ingredients

You know from the earlier sessions what an MDP is: **states, actions, transitions and rewards**.
The insurance contract adds two more ingredients — a cost signal and a budget — and the result is
called a **Constrained Markov Decision Process** (CMDP). This section defines all six on our
mountain, in order:

| | ingredient | in one line |
|---|---|---|
| **1** | states $S$ | where the robot can be |
| **2** | actions $A$ | what it can do |
| **3** | transitions $P$ | where a move takes it |
| **4** | reward $r$ | what a move pays, in CHF |
| **5** | cost $c$ | what a move breaks, in damage points |
| **6** | budget $d$ | how much damage the contract allows |

The first four are the ordinary MDP. Numbers **5** and **6** are what the insurer added.

### 1 · States, $S$

The situations the robot can be in. Here, a situation is simply a position on the map:

$$S \;=\; \{\,(\text{row},\,\text{column})\,\} \qquad |S| = 6 \times 8 = 48$$

We write $s_t$ for the robot's cell after $t$ moves; $s_0$ is the depot.

### 2 · Actions, $A$

What the robot can do in a state. There are four, available everywhere:

$$A \;=\; \{\,\text{Up},\; \text{Right},\; \text{Down},\; \text{Left}\,\}$$

A **policy** picks one action for every state. Choosing a good policy is the entire job.

In [ ]:
rv.fig_states_actions()

### 3 · Transitions, $P$

Where a move takes you, as a probability:

$$P(s' \mid s, a) \;=\; \text{the chance of landing in } s' \text{ after doing } a \text{ in } s$$

The walking itself is deterministic: one cell in the chosen direction, and a fence keeps the
robot on the map at the edges. The randomness is in the ground, and it only ever takes one of
three shapes:

In [ ]:
rv.fig_transitions()

The simulator knows each probability $p$ exactly, and every planner in this notebook will
use them. There are only two such numbers on this map, and the budget section below states
both.

### 4 · Reward, $r$ (in CHF)

What a move pays. This is the familiar signal, and it is fully known:

| when the robot | $r$ |
|---|---|
| makes any move at all | **−1** (effort) |
| steps onto a service-road cell (undermined ones included) | **+1** |
| steps onto a rim cell of the panorama path, heading east | **+5** (tourist tips) |
| arrives at the lodge | **+50** (the delivery bonus) |

### 5 · Cost, $c$ (in damage points)

The first new ingredient: a second number returned on every step, counting harm instead of money.

$$c \;=\; \begin{cases} 100 & \text{if the robot falls on this step}\\[2pt] 0 & \text{otherwise}\end{cases}$$

Both signals arrive together, from the same call:

```python
s_next, r, c, done, info = env.step(a)
```

Side by side:

| | $r$ — reward, in CHF | $c$ — cost, in damage points |
|---|---|---|
| what it measures | what the move **pays** | what the move **breaks** |
| on this map | −1 effort · +1 service road · +5 panorama rim (heading east) · +50 at the lodge | 100 if the robot falls on this step, 0 otherwise |
| stepping east onto the rim, ordinary morning | −1 effort + 5 tips → $r = +4$ | nothing gave way → $c = 0$ |
| the same step, the morning the rim gives way | still $r = +4$ | $c = 100$, and the trip ends |

The last row is the one to remember: **read the reward alone and a catastrophe looks like an
ordinary step.** Nothing in the simulator converts damage points into francs. That conversion is
a management decision, and Part 3 is about who makes it.

Here are both signals painted onto the map — the same 48 cells, first by what they pay,
then by what they can cost:

In [ ]:
rv.fig_reward_cost()

A fall can happen on **14 of the 48 cells**, and those are the cells every safety mechanism in
this notebook is about:

| dangerous cells | which ones | why |
|---|---|---|
| the 4 rim cells | (5,2) (5,3) (5,4) (5,5) | they border the gorge |
| the 2 open pits | (3,2) (3,5) | stepping in *is* falling |
| the 8 undermined cells | (2,2) (2,5) · (3,1) (3,3) (3,4) (3,6) · (4,2) (4,5) | they border a pit |

Two things to keep in mind:

1. **The reward does not register falls.** The step that ends in the gorge pays an ordinary $r$:
   −1 plus whatever tips were collected. Looking only at $r$, a catastrophe is just a short day.
2. **The same slot appears in every industry.** Here it counts falls; in a car it counts
   collisions, in a bank limit breaches, in a hospital adverse events.

### Adding a trip up: $G$ and $G_C$

A trip is scored twice, one running total per signal:

$$G \;=\; r_1 + r_2 + \dots + r_T \qquad\qquad G_C \;=\; c_1 + c_2 + \dots + c_T$$

$G$ is the trip's profit in CHF. $G_C$ is the trip's damage in points, which here is 100 if the
robot fell and 0 if it did not. Standard RL only ever looks at $G$.

| | $G$ — trip profit | $G_C$ — trip damage |
|---|---|---|
| what it answers | what did this delivery earn? | what did this delivery break? |
| who reads it | the board, in the quarterly report | the insurer, against the contract |
| a clean panorama trip | **+63 CHF** — seven moves, four of them scenic, plus the bonus | **0** — the robot came home |
| the same trip, the rim gives way early | **+3 CHF** — it stops before the +50 bonus | **100** |

$G_C$ takes two values on this mountain: **0 or 100**. So the panorama path's *average* damage of
18.55 points describes a trip that never actually happens, which is what §3.5 is about.

### 6 · Budget, $d$

**$d$ is the most average damage per trip the insurer accepts:**

$$\mathbb{E}\big[\,G_C\,\big] \;\le\; d$$

*In plain English: run the deliveries day after day, and the damage must average out to at most
$d$ points per trip.* Nobody computes $d$. It is written into the contract, a **risk-appetite
decision** taken by the board and the insurer, never by the algorithm. GaGGle's contract says
**$d = 5$**.

Since a fall costs exactly 100 points and everything else costs 0, the cap converts directly into
a fall rate:

$$\mathbb{E}\big[\,G_C\,\big] \;=\; 100 \times \mathbb{P}(\text{the robot falls on the trip})$$

*So $d = 5$ allows a route that falls on 5% of its trips, and nothing riskier.*

The cap is about the whole trip, not a single move. A route that crosses $k$ risky cells, each
holding with probability $1-p$, falls with probability $1 - (1-p)^{\,k}$. Small risks compound,
and the route that ended last quarter in the gorge shows how fast:

In [ ]:
rv.fig_route_budgets(which=["panorama"],
                     title="the route that ended last quarter in the gorge")

One 5% crossing would have sat exactly on the cap. Four of them compound to **18.55%**,
which is **3.7 times the agreed limit**. The most profitable route is simply not an option under
this contract, and finding out what to run instead is the work of Parts 2 and 3.

### The objective

Put the six pieces together and the delivery problem becomes one line:

$$\max_{\text{policy}} \;\; \mathbb{E}\big[\,G\,\big] \qquad \text{subject to} \qquad \mathbb{E}\big[\,G_C\,\big] \;\le\; d$$

Find the most profitable way to run deliveries **among the ways that respect the insurance
contract**. An ordinary MDP would stop at the first half; the constraint is the whole difference.

## 1.2 · Forty trips on the panorama path

Now run the route your predecessor's software chose: straight east along the rim. One trip tells
you nothing about risk, so the cell below runs it **forty times**, the first few step by step and
the rest in fast forward. Watch the two running averages, $\mathbb{E}[G]$ and
$\mathbb{E}[G_C]$ — the two quantities the objective is written in.

In [ ]:
env = rv.Trail(seed=9)         # 🔁 change or remove the seed to get different trips
panorama_walk = [1] * 7        # seven moves east: depot -> rim -> lodge

walks = []
for _ in range(40):
    s, done = env.reset(), False
    steps = []
    for a in panorama_walk:
        s, r, c, done, info = env.step(a)
        steps.append({"reward": r, "cost": c, "fell": info["fell"], "state": s})
        if done:
            break
    walks.append(steps)

profits = [sum(step["reward"] for step in walk) for walk in walks]
damages = [sum(step["cost"] for step in walk) for walk in walks]
print(f"{sum(dmg > 0 for dmg in damages)} of 40 trips ended in the gorge")
print(f"average profit {np.mean(profits):+.1f} CHF per trip   ·   "
      f"average damage {np.mean(damages):.1f} points per trip")

rv.walk_marathon(walks)      # ▶ plays on load · ⏭ skip to the end

A wall of green dots at +63, a few red crosses that stopped partway, and an average line
describing **a trip that never actually happens**. Every number your predecessor reported was
that line.

- On the profit chart the red crosses look like nothing much, because the chart is in CHF and the
  100-point hits live in $c$.
- The **average damage** counter is the quantity the contract constrains, and on this route it
  lands nowhere near the cap of 5.

### 🧠 Checkpoint before the code starts

In [ ]:
rv.mc_quiz("cmdp_elements")

In [ ]:
rv.true_false_quiz("reward_vs_cost")

---
# Part 2 — The planner and the shield

Now we compute routes. Everything runs against the digital twin: the map, the payouts and the
fall probabilities, written down as matrices. Given those, finding the most profitable policy is a
solved problem, and any operations-research toolbox ships an algorithm for it. We use one as a
**black box**: first with no safety machinery at all, to reproduce your predecessor's route, then
with the crudest possible fix, a **shield**.

## 2.1 · The mountain as matrices

Three arrays hold everything the planner needs:

In [ ]:
P, R, C = rv.mdp_tensors()

print("P:", P.shape, "— for each of the 4 actions, a 49 x 49 table: where does the move land you?")
print("R:", R.shape, "— expected CHF of each (cell, action) pair")
print("C:", C.shape, "— expected damage points of each (cell, action) pair")
print()
print("(49 = the 48 map cells + one bookkeeping state meaning 'the trip is over')")

Every move the robot could make has an address: the cell it starts from, `s`, and the
direction it takes, `a`. Look that address up in each of the three arrays and you learn where the
move lands, what it pays, and what it can break:

In [ ]:
rv.fig_tensors()

`P` and `R` are the MDP ingredients from §1.1. `C` is the cost ingredient in the same
format: same shape, same indexing, different signal.

| | $R[s,a]$ — the money table | $C[s,a]$ — the damage table |
|---|---|---|
| what it holds | the average $r$ earned by doing $a$ in $s$ | the average $c$ caused by the same move |
| unit | CHF | damage points |
| "step east onto the gorge rim" | **+4.0** — −1 of effort, +5 of tips | **5.0** — a 5% chance of a 100-point fall, averaged out |

**One rim step alone has an expected damage of 5.0, the entire budget $d = 5$.** Every method in
this notebook answers one question: how does `C` get to influence the plan? §2.2 gives the planner
`R` only, §2.5 uses `C` to ban cells outright, Part 3 puts a price on it.

## 2.2 · Planning with the money table only

Now we rebuild your predecessor's software. It used a standard planning algorithm called **policy
iteration**. The planner has the whole map in front of it, so it can try routes on paper instead
of walking them: it works out what each cell is worth, keeps the best move in every cell, and
repeats until nothing changes. What comes out is a **policy**, one chosen move for every cell.

The part that matters here is **what we hand it**. It gets `P`, so it knows where moves lead and
how often the ground gives way. It gets `R`, so it knows what every move pays. It does **not** get
`C`. The damage table stays on the desk, so the planner has one job: earn as many francs as
possible.

> ⚙️ *One technicality: the planner uses a discount factor γ = 0.99, so among equally-paying
> routes it prefers shorter ones. A tie-breaker here, not a dial.*

In [ ]:
policy_neutral = rv.solve_mdp(P, R)      # <- note what we did NOT hand it: C stayed on the desk

print("The planner's verdict:", rv.route_name(policy_neutral))
rv.fig_route_budgets(which=["panorama"], show_math=False,
                     title="what the planner chooses when it is only shown the money")

It sends the robot straight along the gorge rim, which is exactly what it was asked to do.
The panorama pays +5 a stretch, the planner maximises expected profit, and the damage those steps
cause sits in `C`, which it never saw.

Notice what the planner *did* know. The fall probabilities are inside `P`, so it understood that
a rim step sometimes ends the trip early and loses the +50 bonus. It weighed that up and still
chose the rim. **Nothing was hidden from it. The objective was the wrong one.**

## 2.3 · The audit

Before fixing anything, measure. We run the planner's route 2000 times in the simulator and
record both totals per trip. This `evaluate` loop is the auditor for the rest of the notebook —
it simply adds up `r` into one total and `c` into the other:

In [ ]:
def evaluate(policy, n_walks=2000, seed=123):
    """Walk a policy n_walks times. Returns (profit per walk, damage per walk)."""
    env = rv.Trail(seed=seed)
    money, damage = [], []
    for _ in range(n_walks):
        s, done, m, dmg = env.reset(), False, 0.0, 0.0
        while not done:
            s, r, c, done, info = env.step(policy[s])
            m += r
            dmg += c
        money.append(m)
        damage.append(dmg)
    return np.array(money), np.array(damage)

money_neutral, damage_neutral = evaluate(policy_neutral)

print(f"average profit : {money_neutral.mean():+6.1f} CHF per trip")
print(f"average damage : {damage_neutral.mean():6.1f} points per trip")
print(f"trips ending in the gorge: {(damage_neutral > 0).mean():.1%}")

rv.outcome_hist(
    {"RED — the risk-neutral plan (panorama path)": (money_neutral, rv.C_RED)},
    spotlight=(1, 19, "these trips ended in the gorge -\nin the profit chart they only\nlook like slow mornings"))

Nearly a fifth of all trips end with the robot in the gorge, and on the profit axis they are the
small lump on the left, mixed in with the slow days. **The profit histogram has no idea a disaster
happened.** The 100-point hits live in the cost signal, which we plot in Part 3.

## 2.4 · The shield: safety by list

The oldest fix in industrial safety is a **list**: *here are the things you never do, whatever
the spreadsheet says.* In RL this is called **shielding**, and because the filter acts *before*
the action is taken rather than vetoing it afterwards, **preemptive shielding**: unsafe actions
are masked out of the menu, so the planner never considers them.

$$\mathcal{A}_{\text{safe}}(s) \;=\; \{\, a \;:\; \text{the move } a \text{ from } s \text{ does not land on a danger cell} \,\}$$

*In plain English: cross unsafe moves off the menu first; optimise over whatever survives.*

Who writes the list? On this mountain: **you**, from the map, using the rule from Part 1: a cell
is dangerous exactly when it touches a hazard. The hazards are visible even though their
probabilities are not. Try it by hand first:

In [ ]:
rv.shield_lab()

## 2.5 · 🎯 Task 1 — write the shield in code

Same idea, two lines of Python. The map publishes its danger zones as three sets of cells:
`rv.RIM_CELLS` (the gorge rim), `rv.UNDERMINED_CELLS` (the ring around the pits) and
`rv.PIT_CELLS` (the pits themselves), and `rv.next_cell(s, a)` tells you where a move lands. Build
the mask and hand it to the same planner.

In [ ]:
# 🎯 the safety officer's list: every cell where the robot can fall.
#    The map publishes rv.RIM_CELLS, rv.UNDERMINED_CELLS and rv.PIT_CELLS.
#    Combine all three sets                                  (hint: sets combine with | )
UNSAFE_CELLS = ???

action_mask = np.ones((rv.N_STATES, rv.N_ACTIONS), dtype=bool)     # True = allowed
for s in range(rv.N_STATES):
    for a in range(rv.N_ACTIONS):
        # 🎯 mask this move if it lands on an unsafe cell    (hint: rv.next_cell(s, a))
        if ??? in UNSAFE_CELLS:
            action_mask[s, a] = False

print(f"moves masked out: {int((~action_mask).sum())} of {action_mask.size}")
print(f"the shield forbids {len(UNSAFE_CELLS)} cells; the planner will never consider entering them")

Now the exact same planner, on the censored menu:

In [ ]:
policy_shield = rv.solve_mdp(P, R, mask=action_mask)     # same solver, smaller menu

print("The shielded planner's verdict:", rv.route_name(policy_shield))
rv.fig_route_budgets(which=["panorama", "forest"], show_math=False,
                     title="what the shield changed: the planner's route before and after")

money_shield, damage_shield = evaluate(policy_shield)
print(f"risk-neutral : profit {money_neutral.mean():+6.1f} CHF/trip   damage {damage_neutral.mean():5.1f} pts   gorge {(damage_neutral > 0).mean():5.1%}")
print(f"shielded     : profit {money_shield.mean():+6.1f} CHF/trip   damage {damage_shield.mean():5.1f} pts   gorge {(damage_shield > 0).mean():5.1%}")

rv.outcome_hist(
    {"RED — risk-neutral (panorama path)": (money_neutral, rv.C_RED),
     "BLUE — shielded (forest detour)": (money_shield, rv.C_BLUE)})

The robot climbs off the rim, avoids the pits and their ring, and takes the long way through the
forest. Damage drops to **zero, permanently**, and the blue profit distribution collapses to a
single spike at +37. This is your **baseline**: the safest policy this mountain permits, bought
with about 16 CHF of average profit per trip.

**What the shield gets right — and what it cannot do.**

| | |
|---|---|
| ✅ **total** | a masked move is never taken, not merely taken rarely |
| ✅ **auditable** | the whole safety policy is a list of cells anyone can read and challenge |
| ✅ **simple** | you wrote it in two lines, from the map, with no data at all |
| ❌ **binary** | it treats undermined ground, where collapses are rare, exactly like the crumbling rim and the open pits. Off is off, and there is no way to say *“a little of this risk is acceptable”* |
| ❌ **only as good as its list** | a danger that is not on the map never gets masked; a harmless cell someone flags in panic is forbidden forever |
| ❌ **cannot answer the board** | the insurer's letter says *“expected damage ≤ 5 points per trip”* — a **quantity**. The shield only knows *yes* and *no*, so its only answer is the most expensive one: buy **all** the safety, at full price |

That last row is what Part 3 answers.

In [ ]:
rv.mc_quiz("shield_limits")

---
# Part 3 — Choosing a route under the contract

Part 2 gave the company one safe route and one expensive lesson: a shield is all-or-nothing.
Part 3 does the job properly, one board question at a time:

| the question | what this section builds | where |
|---|---|---|
| Which routes could we actually run? | three concrete options, priced in profit *and* damage | §3.1 |
| Which of them does the contract allow? | the damage budget **$d$** | §3.2 |
| How does the planner respect the cap on its own? | the internal price **$\lambda$** | §3.3 |
| Where does that price come from? | **dual ascent** from a stated budget | §3.4 |
| How bad are the bad days? | the outcome distribution and **CVaR at level $\alpha$** | §3.5 |

All three dials are set by people, and §3.6 puts them side by side.

## 3.1 · Three ways to run the deliveries

The shield from Part 2 is really a **list of banned cells**, and different lists give different
routes. Three of them are worth putting to a board:

- **ban nothing** — run wherever it pays best,
- **ban the gorge rim and the pits** — stay off the worst ground, accept the rest,
- **ban every dangerous cell** — the fully shielded route from Part 2.

Same planner, same map, three proposals:

In [ ]:
def shield_mask(forbidden):
    """Mask every move that would land on a cell in `forbidden`."""
    mask = np.ones((rv.N_STATES, rv.N_ACTIONS), dtype=bool)
    for s in range(rv.N_STATES):
        for a in range(rv.N_ACTIONS):
            if rv.next_cell(s, a) in forbidden:
                mask[s, a] = False
    return mask

PROPOSALS = [("ban nothing",                    frozenset()),
             ("ban the gorge rim and the pits", rv.RIM_CELLS | rv.PIT_CELLS),
             ("ban every dangerous cell",       rv.DANGER_CELLS)]

candidates = []
print(f"{'the safety list':<32} {'route it produces':<22} {'profit':>8} {'damage':>8} {'falls':>7}")
for label, forbidden in PROPOSALS:
    policy = rv.solve_mdp(P, R, mask=shield_mask(forbidden))
    dist = rv.trip_distribution(policy)            # exact, straight from the model
    candidates.append({"name": rv.route_name(policy), "policy": policy, "dist": dist,
                       "avg_money": dist["mean_money"], "avg_damage": dist["mean_damage"],
                       "fall_pct": 100 * dist["p_fall"]})
    print(f"{label:<32} {rv.route_name(policy):<22} {dist['mean_money']:>+8.2f} "
          f"{dist['mean_damage']:>8.2f} {dist['p_fall']:>7.2%}")

by_name = {cand["name"]: cand for cand in candidates}

**Three genuine business options.** The panorama earns the most and breaks the most, the forest
detour is the mirror image, the old service road sits in between. Each one below is drawn with its
risky cells circled and its damage worked out by the compounding rule from Part 1:

In [ ]:
rv.fig_route_budgets()

These are three priced positions on the same mountain, not one safe route and two unsafe
ones. Which of them the company should run is the next question.

## 3.2 · The board's budget, $d$

Ingredient 6 from §1.1 decides between them. The contract caps expected damage at $d = 5$ points
per trip, and against a shortlist of three that cap is a two-step rule:

> **Keep the routes whose expected damage is at most $d$. Among those, choose the most profitable
> one.**

Drag the cap below and watch the shortlist change:

In [ ]:
rv.budget_lab(candidates, budget_init=5.0)

**At the contract's $d = 5$ the answer is the old service road.** The panorama path spends
18.55 and is disqualified however much it earns; the forest detour and the service road both fit,
and the service road earns **+47.06** against the forest detour's **+37.00**. Ten francs a trip,
for staying inside the same contract.

### 🎯 Task 2 — the same rule, as a board summary

A board will ask the obvious follow-up: *what would a different cap have bought us?* That is the
same two-step rule, run for a range of caps.

In [ ]:
D_MENU = [0, 1, 3, 5, 10, 15, 20, 25]

menu = []
for d in D_MENU:
    # 🎯 which of the candidate routes fit this budget?    (hint: cand["avg_damage"] <= d)
    affordable = [cand for cand in candidates if ???]
    best = max(affordable, key=lambda cand: cand["avg_money"])
    menu.append(best)
    print(f"budget d = {d:>4} :  best affordable route = {best['name']:<22s}"
          f"  expected profit {best['avg_money']:+6.2f} CHF")

rv.budget_menu_plot(D_MENU, menu)

A price list for safety: tightening the cap from 25 to 5 costs about 5.9 CHF a trip, and
tightening it from 5 to 0 costs another 10.1. The board is choosing how much profit per trip to
pay for a lower fall probability.

## 3.3 · 🎯 Task 3 — the internal price, $\lambda$

Filtering a list of three works because *we* wrote the list. A planner cannot do that: it has 48
cells and four actions everywhere, so it needs **one number to maximise**.

> **$d$ is the board's damage budget — a promise about outcomes.**
> **$\lambda$ is the internal price the optimiser uses — CHF charged per damage point.**

Charge the planner λ CHF for every expected damage point and it can compare profit and risk on a
single scale:

$$\text{score} \;=\; \text{expected profit} \;-\; \lambda \times \text{expected damage}$$

*In plain English: take the profit, then subtract a fine of λ CHF for every damage point the
route is expected to cause. Whatever scores highest is the plan.*

Applied to a single move, the same idea is called **reward shaping**: the planner is handed
$\tilde r = r - \lambda\,c$ instead of $r$. Nobody pays this fine. It is an accounting price the
company charges itself so that francs and damage points become one comparable number, and §3.4
connects it back to the budget.

At $\lambda = 1.5$ CHF per damage point, the three routes score like this:

In [ ]:
rv.fig_risk_charge(candidates, lam=1.5)

The panorama earns the most and still loses. Its 18.55 damage points are charged at
1.5 CHF each, so 27.8 CHF comes off its score: 52.9 − 27.8 = **25.1**. The service road is charged
only 1.5 × 3.96 = 5.9, leaving **41.1**, and the forest detour scores its unchanged **37.0**.
**Nobody told the planner about the budget** — the price alone made it pick the route the contract
allows.

In code, the price is one line: charge every move for the damage it is expected to cause, then
hand the shaped numbers to the same planner as before.

In [ ]:
LAMBDAS = [0.0, 0.4, 1.5, 4.0]

print(f"{'λ':>4}  {'route the planner picks':<26} {'profit':>9} {'damage':>8}")
for lam in LAMBDAS:
    # 🎯 the shaped money menu: money minus lambda CHF per expected damage point
    #    (hint: R, lam, C — it looks exactly like the equation above)
    R_shaped = ???
    policy_lam = rv.solve_mdp(P, R_shaped)      # the same risk-blind planner as Part 2

    dist = rv.trip_distribution(policy_lam)
    print(f"{lam:>4.1f}  {rv.route_name(policy_lam):<26} {dist['mean_money']:>+9.2f} "
          f"{dist['mean_damage']:>8.2f}")

A cheap price leaves the panorama optimal; an expensive one drives the planner all the way
to the forest. Because the world is small, we can find every switch point exactly:

In [ ]:
fine = np.arange(0.0, 5.0, 0.0001)
picks = [rv.route_name(rv.solve_mdp(P, R - lam * C)) for lam in fine]

switches, bands = [], [picks[0]]
for lam, name, prev in zip(fine[1:], picks[1:], picks[:-1]):
    if name != prev:
        switches.append(round(float(lam), 4)); bands.append(name)
print("the planner changes its mind at λ =", switches)

rv.lambda_ladder_plot(switches, bands)

The whole λ dial on one axis:

| the price the company charges itself | what the planner does with it |
|---|---|
| $\lambda < 0.4804$ | **Risk is cheap.** The tips on the rim outweigh the damage charge, so the planner keeps running the panorama path — 18.55 damage points, well over the contract. |
| $0.4804 \le \lambda \le 3.165$ | **Risk is priced about right.** The planner picks the service road by itself: the route the $d = 5$ contract wants, without ever being told that a contract exists. |
| $\lambda > 3.165$ | **Risk is expensive.** Even the rarely-collapsing undermined ground is not worth crossing, and the planner retreats to the forest detour and gives up 10 CHF a trip. |

So the two dials are connected: **pick the budget, and some price enforces it.** The middle band
is wide, so the price does not have to be exact, only inside it. The next section finds it without
guessing.

*Those switch points belong to our planner, which discounts slightly (γ = 0.99); crossovers
computed from the undiscounted table above land a little differently.*

## 3.4 · 🎯 Task 4 — letting the price find itself

Nobody wants to guess λ. So state the budget instead, and let the price adjust:

1. plan at the current price λ,
2. measure the damage it produced,
3. **damage above budget → raise λ. Damage below budget → lower λ. Never below zero.**

That loop is called **dual ascent**, and it has one more part: the board reacts **less strongly
every meeting**.

$$\lambda \;\leftarrow\; \max\Big(0,\;\; \lambda \;+\; \underbrace{\eta_0 \cdot \tfrac{1}{1+t}}_{\text{the fading reaction }\eta_t}\big(\text{measured damage} \;-\; d\big)\Big)$$

*In plain English: overspending makes risk more expensive, underspending makes it cheaper, the
price never goes negative (that would pay the robot to fall), and each correction is smaller than
the one before it.*

**Why the reaction has to fade.** Keep it constant and the price never settles: it jumps up, the
fleet retreats to the service road, the price slides down, the panorama becomes attractive again,
and the loop swings between the two forever. The loop is fine; the problem simply has no
single-route answer. At $d = 5$ the exact optimum is a **randomized operating mix** —
run the service road about **93%** of the time and the panorama the other **7%**, so the average
damage lands exactly on 5. A planner that must hand back one route cannot express a mix, so a
constant step keeps hunting for it. A fading step lets the price converge anyway, and little is
lost: the mix would earn **+47.48** against the service road's **+47.06**, about **0.4 CHF a
trip**.

In [ ]:
BUDGET    = 5.0                              # the insurer's cap, in damage points per trip
ETA_LAM   = 0.05                             # how hard the board reacts to being off budget
LR_LAMBDA = lambda iteration: 1 / (1 + iteration)   # …and how fast that reaction fades

lam, lam_history, damage_history = 0.0, [], []
for meeting in range(50):
    policy_round = rv.solve_mdp(P, R - lam * C)                      # ops re-plans at price λ
    avg_damage = rv.trip_distribution(policy_round)["mean_damage"]   # the auditor measures
    lam_history.append(lam); damage_history.append(avg_damage)

    if meeting < 8 or (meeting + 1) % 10 == 0:                       # print a readable sample
        verdict = "over budget" if avg_damage > BUDGET else "within budget"
        print(f"meeting {meeting + 1:>2}:  λ = {lam:5.3f}  →  {rv.route_name(policy_round):<20s}"
              f"  damage {avg_damage:5.2f}   ({verdict})")

    # 🎯 the board moves the price: up when over budget, down when under, never below zero
    #    (hint: max(0.0, lam + ETA_LAM * LR_LAMBDA(meeting) * (avg_damage - BUDGET)))
    lam = ???

def find_settled_lambda(lam_history):
    # We select the smallest λ that is not followed by a larger λ
    # because if a round raised λ, it means the constraint was violated
    non_overshoot = []
    lams = lam_history[1:]
    for i in range(len(lams)-1):
        if lams[i]>=lams[i+1]:
            non_overshoot.append(lams[i])
    return min(non_overshoot) if non_overshoot else None

LAMBDA = find_settled_lambda(lam_history)
print(f"\nthe price has settled at λ = {LAMBDA}; the fleet runs {rv.route_name(policy_round)} "
      f"at {avg_damage:.2f} damage points, inside the budget of {BUDGET:g}.")

rv.dual_ascent_plot(lam_history, damage_history, BUDGET)

The price starts at zero, the robot takes the panorama, and 18.55 against a budget of 5 is
a large miss — so the first correction is big, and the fleet is on the service road by the second
meeting. From there the damage sits *under* budget, so the price eases back down, each nudge
smaller than the one before, closing in on **0.485**: the switch point from §3.3, the cheapest
price that still buys the service road. **The board stated a budget and got a price, without
anyone guessing one.**

In [ ]:
rv.mc_quiz("price_vs_budget")

## 3.5 · The third dial: how bad are the bad days?

Every number so far has been an **average**. But no trip ever earns +47.06: that is the average
of a few very different days, and we can print them exactly, because we own the model:

In [ ]:
for cand in candidates:
    print(f"{cand['name']}:")
    for profit, prob in sorted(cand["dist"]["money"].items()):
        print(f"    {prob:6.2%} of trips end up paying {profit:+6.1f} CHF")
    print(f"    → average {cand['dist']['mean_money']:+.2f} CHF\n")

A route is a **distribution** of days, not a number. Computing it exactly instead of sampling
trips is called *distributional policy evaluation*, and it lets you ask sharper questions than
"what is the average". Both quantities as distributions:

In [ ]:
rv.outcome_panel({name: (cand["dist"], rv.ROUTE_COLORS[name])
                  for name, cand in by_name.items()}, budget=5)

Look at the damage panel: a trip breaks **either nothing or everything**, so the average of
18.55 for the panorama describes a trip that never happens.

The budget caps that average and the price charges for it. **Both read one number off the
distribution and throw the rest away**, which is what Part 0's two offers were about.

So here is the third question: **forget averages, how bad are my bad days?** The standard answer
is **CVaR** (Conditional Value at Risk), and it has one dial.

$$\text{CVaR}_\alpha(G)=\mathbb{E}[G\hspace{2pt}|\hspace{2pt}G\geq \text{VaR}_\alpha(G)]=\; \text{the average outcome for } G \text{ over the worst } \alpha \text{ share of trips}$$

*In plain English: line every trip up worst first, keep the worst α share of them, and average
only those.* The rest of the distribution is deliberately thrown away.

**$\alpha$ is the risk level: the share of worst trips your report looks at.** It describes the
question you ask, not the mountain, which makes it a management decision like $d$ and $\lambda$.
Setting $\alpha = 0.05$ is the instruction *"report the worst trip in twenty, and ignore the other
nineteen."*

| the level you report at | the question it asks | who wins here |
|---|---|---|
| $\alpha = 1.00$ | Every trip counts equally — this *is* the plain average, your predecessor's objective. | the panorama path |
| $\alpha = 0.20$ | How do the worst trips in five look? A normal risk report. | the service road |
| $\alpha \to 0$ | What is the single worst thing that can happen? Pure disaster-avoidance. | the forest detour |

Worked example, the service road at $\alpha = 0.05$: its worst 3.96% of trips pay 0 CHF and the
rest of the 5% window pays +49, so $\text{CVaR}_{0.05}(G) = +10.19$ CHF against a plain average of
+47.06. Same route, same model, a very different number to report.

> **Sign convention.** We maximise profit, so the bad tail is the *low* one. Texts written in cost
> terms minimise, and their bad tail is the *high* one. Multiply by −1 and the mathematics is
> identical.

### 🎯 Task 5 — CVaR from an exact distribution

The distribution is a set of outcomes with probabilities, so "the worst α share" may cut an
outcome in half: if a trip type has probability 0.05 and α = 0.02, only *part* of it is inside
the window. Walk the outcomes from worst to best and take as much of each as still fits.

In [ ]:
def cvar(distribution, alpha):
    """Average outcome over the worst `alpha` share of trips.

    distribution : {outcome value: probability}, e.g. {3.0: 0.05, 63.0: 0.81}
    alpha        : 1.0 = the plain mean, 0.05 = the worst 5% of trips
    """
    total, taken = 0.0, 0.0
    for value, prob in sorted(distribution.items()):      # worst outcomes first
        # 🎯 how much of this outcome's probability still fits in the window?
        #    never more than what is left of alpha      (hint: min)
        share = ???
        if share <= 0:
            break
        # 🎯 add this outcome, weighted by the share of the window it takes up
        total += ???
        taken += share
    return total / alpha

# quick check
dummy_distribution = {3.0: 0.05, 10.0: 0.15, 50.0: 0.80}
print(f"CVaR of dummy distribution at α = 0.05: {cvar(dummy_distribution, 0.05):.2f} (should be 3.00)")
print(f"CVaR of dummy distribution at α = 0.1: {cvar(dummy_distribution, 0.1):.2f} (should be 6.50)")
print(f"CVaR of dummy distribution at α = 1.0: {cvar(dummy_distribution, 1.0):.2f} (should be 41.65)")

Now ask all three routes the same question, at three different levels of paranoia:

In [ ]:
dists = {cand["name"]: cand["dist"] for cand in candidates}

print(f"{'α':>6}  " + "  ".join(f"{name:>18}" for name in dists) + "     the winner")
for alpha in (0.02, 0.05, 0.20, 0.50, 1.00):
    scores = {name: cvar(dist["money"], alpha) for name, dist in dists.items()}
    winner = max(scores, key=scores.get)
    print(f"{alpha:>6.2f}  " + "  ".join(f"{scores[name]:>18.2f}" for name in dists)
          + f"     {winner}")

In [ ]:
rv.cvar_lab({name: (dist, rv.ROUTE_COLORS[name]) for name, dist in dists.items()},
            alpha_init=0.20)

**Drag the slider and watch the ranking change.** Three things to notice:

1. **At α = 100% the panorama wins.** α = 1 is the mean, so this is your predecessor's answer.
2. **Around α ≈ 0.58 the service road takes the lead, and around α ≈ 0.16 the forest takes it.**
   Same three routes, same map, nothing retrained: only the question changed.
3. **Below α = 4% the panorama overtakes the service road again.** Its worst days pay +3, because
   the robot banked a few tips before the rim gave way; the service road's worst day pays
   **exactly 0**, because it fails before earning anything.

The third point is why this section exists. The service road beats the panorama *only in a middle
band of risk appetite*, a preference that flips **twice**, and neither the budget nor the price
can express that, because both read only a mean.

In [ ]:
rv.cvar_curve(cvar, {name: (dist, rv.ROUTE_COLORS[name]) for name, dist in dists.items()})

> 🔬 **What we did and did not do.** We computed each policy's exact return distribution and
> *scored* it with a risk measure: distributional **evaluation**, then a choice between three
> routes. We did not search all policies for the CVaR-optimal one, because **CVaR is not
> time-consistent**: the CVaR-optimal policy depends on how much risk has already been used, so
> finding it needs the state augmented with a running risk budget, or a policy gradient on a
> learned distributional critic.

## 3.6 · Three dials, one board

None of the three is discovered by the algorithm on its own: the board states $d$ and $\alpha$
outright, and $\lambda$ follows from $d$ once dual ascent is pointed at it. The column that
matters is the fourth one: two of them read a single number off the outcome distribution, and one
reads its shape.

| dial | what it says | what it is | what it reads off the distribution | why you would reach for it |
|---|---|---|---|---|
| $d$ | expected damage per trip must stay under $d$ | a promise about outcomes, in the insurer's own language | the **mean** of the damage distribution, nothing else | easy to audit and to write into a contract |
| $\lambda$ | every damage point costs the planner $\lambda$ CHF internally | a steering signal the optimiser can actually use | the **mean** again, just multiplied by a number | easy to set automatically — dual ascent finds it from $d$ |
| $\alpha$ | score the policy on its worst $\alpha$ share of trips | a question about the tail, not about the average | the **whole shape** of the distribution | the only dial that can separate two policies whose means agree |

Two checks: one on the dials, one on using them in an office.

In [ ]:
rv.true_false_quiz("three_dials")

In [ ]:
rv.mc_quiz("transfer")

---
# 🎓 Wrap-up

One mountain, three plans, and the core ideas of risk-aware decision-making in miniature:

| the object | what it is | why it matters |
|---|---|---|
| Reward and cost | two separate signals from every step: reward $r$ in CHF, cost $c$ in damage points | danger is measured on its own; nobody converts it into money without a decision |
| The constrained MDP | maximise expected profit subject to expected damage $\le d$ | the insurance contract, written as mathematics |
| Preemptive shielding | a hand-made list of forbidden cells masks actions before the planner sees them | safety by list: total and auditable, but one switch with no dial |
| The planner | off-the-shelf policy iteration, used as a black box | given the model, finding the most profitable route is a solved problem |
| The outcome distribution | computed exactly from the model: every profit and damage a route can produce, with its probability | the return of a policy is a distribution; the mean is one summary of it |
| CVaR at level $\alpha$ | the average over the worst $\alpha$ share of trips; $\alpha = 1$ is the plain mean | the only dial that reads the shape of the tail instead of a single average |
| The $\lambda$ price and dual ascent | plan with $r - \lambda c$; raise $\lambda$ while damage exceeds $d$, lower it below, react more gently each round | the budget is chosen by management; the price finds itself and settles |

### 🔖 Every symbol in this notebook

| symbol | name | what it means |
|---|---|---|
| $r$ | reward | what one move pays, in CHF: effort, tips, the delivery bonus |
| $c$ | cost | damage points from one move: 100 on a fall, else 0 — separate from reward |
| $G$, $G_C$ | trip totals | one trip's profit (sum of $r$) and one trip's damage (sum of $c$) |
| $P$, $R$, $C$ | the model | transition probabilities, expected reward and expected cost per move |
| $d$ | the budget | the expected damage per trip management will sign — **the board's number** |
| $\lambda$ | price of risk | internal CHF per damage point; found by dual ascent, anchored by $d$ |
| $\alpha$ | the risk level | which share of the **worst** trips your score looks at; $\alpha = 1$ is the plain mean |
| $\eta_t$ | the reaction speed | how hard meeting $t$ moves the price; decays as $\eta_0/t$ so the price settles |
| $\gamma$ | discount | the solver's tiny impatience (0.99) — a tie-breaker here, not a dial |

The four equations, in the order you built them:

$$\underbrace{\max \; \mathbb{E}[G] \;\; \text{s.t.} \;\; \mathbb{E}[G_C] \le d}_{\text{the promise (CMDP)}}
\qquad
\underbrace{\tilde r = r - \lambda\, c}_{\text{the price}}$$

$$\underbrace{\lambda \leftarrow \max\!\big(0,\; \lambda + \eta_t(\text{measured} - d)\big)}_{\text{the discovery loop}}
\qquad
\underbrace{\text{CVaR}_\alpha = \text{mean of the worst } \alpha \text{ share}}_{\text{the tail question}}$$

**Where you will meet this again, outside a mountain:**

- **Autonomous driving** — a learned safety critic in the role of our `C`, constraints enforced
  during training, λ found by dual ascent.
- **Trading & treasury** — regulators cap risk measures of the P&L distribution (Basel's expected
  shortfall is CVaR); a trading desk's internal cost of risk capital plays the role of λ.
- **Credit & insurance** — approval systems with an exposure budget per segment: a separate cost
  signal with a cap, and a price that steers the optimiser under it.
- **Healthcare dosing, energy grids, supply chains** — the same pattern wherever a good average
  can hide a rare disaster.

## 🏁 Final check — the whole notebook

In [ ]:
rv.true_false_quiz("wrapup")